In [ ]:
import torch
import matplotlib.pyplot as plt
from torch import nn
from torchvision.models import convnext_base

import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit


In [ ]:
# Import du dataset numpy
directory = "C:/Users/lrozier/Documents/UQAC/respiratory-disease-detection/data/processed/"
X_features = np.load(directory + "X_features.npy")
X_mel = np.load(directory + "X_mel.npy")
y = np.load(directory + "y.npy")

# Split en train, validation et test
X_train, X_temp, y_train, y_temp = StratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=42).split(X_mel, y)
X_val, X_test, y_val, y_test = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=42).split(X_temp, y_temp)

In [ ]:
# Chargement du modèle ConvNext pré-entraîné
model = convnext_base(pretrained=True)

# Modification de la dernière couche pour correspondre au nombre de classes de notre problème
num_classes = 5
model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)
model = model.to('cuda' if torch.cuda.is_available() else 'cpu')

# Définition de la fonction de perte et de l'optimiseur
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Entraînement du modèle
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for i in range(len(X_train)):
        inputs = torch.tensor(X_train[i], dtype=torch.float32).unsqueeze(0).to('cuda' if torch.cuda.is_available() else 'cpu')
        labels = torch.tensor(y_train[i], dtype=torch.long).unsqueeze(0).to('cuda' if torch.cuda.is_available() else 'cpu')

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()



c:\Users\lrozier\Documents\UQAC\respiratory-disease-detection\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\lrozier\Documents\UQAC\respiratory-disease-detection\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ConvNeXt_Base_Weights.IMAGENET1K_V1`. You can also use `weights=ConvNeXt_Base_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/convnext_base-6075fbad.pth" to C:\Users\lrozier/.cache\torch\hub\checkpoints\convnext_base-6075fbad.pth


100%|██████████| 338M/338M [00:03<00:00, 97.4MB/s] 
